# Distributed Tensors

This notebook covers working with DTL distributed tensors (N-dimensional arrays).

## Topics
- Creating tensors with different shapes
- Understanding tensor partitioning
- Working with multi-dimensional local views
- Common use cases (images, matrices)

In [ ]:
import dtl
import numpy as np

## Creating Tensors

Use `DistributedTensor` to create N-dimensional distributed arrays:

In [ ]:
with dtl.Context() as ctx:
    # 2D tensor (matrix)
    matrix = dtl.DistributedTensor(ctx, shape=(100, 64), dtype=np.float64)
    print(f"2D tensor: shape={matrix.shape}, ndim={matrix.ndim}")
    
    # 3D tensor
    cube = dtl.DistributedTensor(ctx, shape=(10, 20, 30), dtype=np.float64)
    print(f"3D tensor: shape={cube.shape}, ndim={cube.ndim}")
    
    # 4D tensor (e.g., batch of images)
    batch = dtl.DistributedTensor(ctx, shape=(64, 3, 224, 224), dtype=np.float32)
    print(f"4D tensor: shape={batch.shape}, ndim={batch.ndim}")

## Tensor Partitioning

Tensors are partitioned along the **first dimension**. Other dimensions remain intact.

In [ ]:
with dtl.Context() as ctx:
    # Create a 100x64 matrix distributed across ranks
    tensor = dtl.DistributedTensor(ctx, shape=(100, 64), dtype=np.float64)
    
    print(f"Global shape: {tensor.shape}")
    print(f"Local shape: {tensor.local_shape}")
    print(f"\nGlobal size: {tensor.global_size}")
    print(f"Local size: {tensor.local_size}")
    
    # Notice: second dimension (64) is preserved
    print(f"\nNote: First dim distributed, second preserved")

## Working with Local Views

The `local_view()` returns a NumPy array with the local shape:

In [ ]:
with dtl.Context() as ctx:
    tensor = dtl.DistributedTensor(ctx, shape=(100, 64), dtype=np.float64)
    
    local = tensor.local_view()
    print(f"Local view type: {type(local)}")
    print(f"Local view shape: {local.shape}")
    print(f"Local view dtype: {local.dtype}")
    
    # Fill with pattern
    for i in range(local.shape[0]):
        local[i, :] = i + ctx.rank * 100
    
    print(f"\nFirst row: {local[0, :5]}")
    print(f"Last row: {local[-1, :5]}")

## Example: Distributed Matrix

Create and initialize a distributed matrix:

In [ ]:
with dtl.Context() as ctx:
    # 1000x100 matrix
    rows, cols = 1000, 100
    matrix = dtl.DistributedTensor(ctx, shape=(rows, cols), dtype=np.float64)
    
    local = matrix.local_view()
    
    # Initialize with row index in first column, random in others
    np.random.seed(42 + ctx.rank)
    local[:, 0] = np.arange(local.shape[0])  # Row indices
    local[:, 1:] = np.random.randn(local.shape[0], cols - 1)
    
    # Compute local statistics
    print(f"Rank {ctx.rank}:")
    print(f"  Local rows: {local.shape[0]}")
    print(f"  Row mean: {local.mean(axis=1)[:3]}...")
    print(f"  Col mean: {local.mean(axis=0)[:3]}...")

## Example: Batch of Images

Simulate a batch of images distributed across ranks:

In [ ]:
with dtl.Context() as ctx:
    # Batch of 64 RGB images, 224x224 pixels
    batch_size = 64
    channels = 3
    height = 224
    width = 224
    
    images = dtl.DistributedTensor(
        ctx, 
        shape=(batch_size, channels, height, width),
        dtype=np.float32
    )
    
    local = images.local_view()
    
    print(f"Global batch: {images.shape}")
    print(f"Local batch: {local.shape}")
    print(f"Images per rank: {local.shape[0]}")
    
    # Initialize with random pixel values [0, 1]
    np.random.seed(ctx.rank)
    local[:] = np.random.rand(*local.shape).astype(np.float32)
    
    # Compute per-image mean (across channels and pixels)
    per_image_mean = local.mean(axis=(1, 2, 3))
    print(f"Per-image means (first 3): {per_image_mean[:3]}")

## Fill Operations

In [ ]:
with dtl.Context() as ctx:
    # Create with fill value
    tensor = dtl.DistributedTensor(
        ctx, shape=(10, 10), dtype=np.float64, fill=0.0
    )
    
    local = tensor.local_view()
    print(f"After creation with fill=0.0:")
    print(local[:2, :5])
    
    # Fill with new value
    tensor.fill(1.0)
    print(f"\nAfter fill(1.0):")
    print(local[:2, :5])

## Supported Data Types

In [ ]:
with dtl.Context() as ctx:
    dtypes = [np.float64, np.float32, np.int64, np.int32]
    
    for dtype in dtypes:
        tensor = dtl.DistributedTensor(ctx, shape=(10, 10), dtype=dtype)
        local = tensor.local_view()
        print(f"{dtype.__name__}: local_view dtype = {local.dtype}")